# Portfolio Optimization with Modern Portfolio Theory

**Category:** 11-Financial  
**Description:** Mean-variance optimization, efficient frontier, Sharpe ratio  
**Libraries:** NumPy, Pandas, SciPy, Matplotlib

---

In [ ]:
# =============================================================================
# DEPENDENCIES & MODEL ALIASES
# =============================================================================
%pip install -q numpy pandas scipy matplotlib yfinance

# Model aliases
CLAUDE = "anthropic-chat:claude-sonnet-4-5-20250929"
CLAUDE_FAST = "anthropic-chat:claude-haiku-4-5-20251001"
GPT = "openai-chat:gpt-5"
GEMINI = "gemini:gemini-2.5-flash"

MODEL = CLAUDE

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

---

# Part 1: Generate Simulated Asset Returns

---

In [ ]:
# Simulated stock data (realistic parameters)
n_assets = 5
n_days = 252 * 3  # 3 years of daily data
asset_names = ['Tech', 'Healthcare', 'Finance', 'Energy', 'Consumer']

# Expected annual returns and volatilities
expected_returns = np.array([0.12, 0.08, 0.10, 0.06, 0.09])  # 12%, 8%, etc.
volatilities = np.array([0.25, 0.18, 0.22, 0.30, 0.15])       # 25%, 18%, etc.

# Correlation matrix (realistic cross-asset correlations)
corr_matrix = np.array([
    [1.00, 0.35, 0.45, 0.20, 0.50],  # Tech
    [0.35, 1.00, 0.30, 0.15, 0.40],  # Healthcare
    [0.45, 0.30, 1.00, 0.35, 0.45],  # Finance
    [0.20, 0.15, 0.35, 1.00, 0.25],  # Energy
    [0.50, 0.40, 0.45, 0.25, 1.00]   # Consumer
])

# Build covariance matrix
cov_matrix = np.outer(volatilities, volatilities) * corr_matrix

# Generate daily returns
daily_returns = expected_returns / 252
daily_vol = volatilities / np.sqrt(252)
daily_cov = cov_matrix / 252

# Simulate returns using Cholesky decomposition
L = np.linalg.cholesky(daily_cov)
random_returns = np.random.randn(n_days, n_assets)
returns = daily_returns + random_returns @ L.T

# Create DataFrame
dates = pd.date_range('2022-01-01', periods=n_days, freq='B')
returns_df = pd.DataFrame(returns, index=dates, columns=asset_names)

print("Simulated Daily Returns:")
returns_df.head()

In [ ]:
# Calculate cumulative returns (price series)
prices = (1 + returns_df).cumprod() * 100  # Starting price = 100

# Plot price evolution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

prices.plot(ax=axes[0], linewidth=1.5)
axes[0].set_title('Simulated Asset Prices (3 Years)')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Price ($)')
axes[0].legend(loc='upper left')

returns_df.boxplot(ax=axes[1])
axes[1].set_title('Daily Returns Distribution by Asset')
axes[1].set_ylabel('Daily Return')

plt.tight_layout()
plt.show()

# Summary statistics
print("\nAnnualized Statistics:")
summary = pd.DataFrame({
    'Annual Return': returns_df.mean() * 252,
    'Annual Volatility': returns_df.std() * np.sqrt(252),
    'Sharpe Ratio': (returns_df.mean() * 252) / (returns_df.std() * np.sqrt(252))
})
print(summary.round(4))

---

# Part 2: Portfolio Metrics

---

In [ ]:
def portfolio_return(weights, returns):
    """Calculate annualized portfolio return."""
    return np.sum(returns.mean() * weights) * 252

def portfolio_volatility(weights, returns):
    """Calculate annualized portfolio volatility."""
    return np.sqrt(np.dot(weights.T, np.dot(returns.cov() * 252, weights)))

def portfolio_sharpe(weights, returns, risk_free_rate=0.02):
    """Calculate portfolio Sharpe ratio."""
    ret = portfolio_return(weights, returns)
    vol = portfolio_volatility(weights, returns)
    return (ret - risk_free_rate) / vol

# Test with equal weights
equal_weights = np.array([0.2, 0.2, 0.2, 0.2, 0.2])

print("Equal-Weight Portfolio Metrics:")
print(f"  Expected Return: {portfolio_return(equal_weights, returns_df):.2%}")
print(f"  Volatility:      {portfolio_volatility(equal_weights, returns_df):.2%}")
print(f"  Sharpe Ratio:    {portfolio_sharpe(equal_weights, returns_df):.3f}")

In [ ]:
%%ai $MODEL
Explain the key concepts in Modern Portfolio Theory:

1. What is the Sharpe Ratio and why is it important?
2. What does "diversification" really mean mathematically?
3. Why can't we just pick the highest-returning asset?

---

# Part 3: Efficient Frontier

---

In [ ]:
def generate_random_portfolios(returns_df, n_portfolios=5000):
    """Generate random portfolios for visualization."""
    n_assets = len(returns_df.columns)
    results = np.zeros((n_portfolios, 3))  # return, vol, sharpe
    weights_record = []
    
    for i in range(n_portfolios):
        # Random weights that sum to 1
        weights = np.random.random(n_assets)
        weights /= np.sum(weights)
        weights_record.append(weights)
        
        # Calculate metrics
        results[i, 0] = portfolio_return(weights, returns_df)
        results[i, 1] = portfolio_volatility(weights, returns_df)
        results[i, 2] = portfolio_sharpe(weights, returns_df)
    
    return results, weights_record

# Generate random portfolios
results, weights_record = generate_random_portfolios(returns_df, 10000)

# Plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(results[:, 1], results[:, 0], 
                      c=results[:, 2], cmap='viridis', 
                      alpha=0.5, s=10)
plt.colorbar(scatter, label='Sharpe Ratio')

# Mark individual assets
for i, name in enumerate(asset_names):
    ret = returns_df[name].mean() * 252
    vol = returns_df[name].std() * np.sqrt(252)
    plt.scatter(vol, ret, marker='*', s=300, c='red', edgecolors='black', linewidths=1)
    plt.annotate(name, (vol, ret), fontsize=10, ha='left')

plt.title('Random Portfolios & Efficient Frontier Region')
plt.xlabel('Volatility (Annual)')
plt.ylabel('Expected Return (Annual)')
plt.grid(True, alpha=0.3)
plt.show()

---

# Part 4: Optimization

---

In [ ]:
def optimize_portfolio(returns_df, objective='sharpe', target_return=None):
    """
    Optimize portfolio weights.
    
    Args:
        objective: 'sharpe' (max Sharpe), 'min_vol' (min volatility), 'target' (target return)
        target_return: Required if objective='target'
    """
    n_assets = len(returns_df.columns)
    init_weights = np.array([1/n_assets] * n_assets)
    
    # Constraints: weights sum to 1
    constraints = [{'type': 'eq', 'fun': lambda x: np.sum(x) - 1}]
    
    # Bounds: no short selling (0 <= weight <= 1)
    bounds = tuple((0, 1) for _ in range(n_assets))
    
    if objective == 'sharpe':
        # Maximize Sharpe (minimize negative Sharpe)
        result = minimize(
            lambda w: -portfolio_sharpe(w, returns_df),
            init_weights,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints
        )
    elif objective == 'min_vol':
        # Minimize volatility
        result = minimize(
            lambda w: portfolio_volatility(w, returns_df),
            init_weights,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints
        )
    elif objective == 'target':
        # Minimize volatility for target return
        constraints.append({
            'type': 'eq',
            'fun': lambda w: portfolio_return(w, returns_df) - target_return
        })
        result = minimize(
            lambda w: portfolio_volatility(w, returns_df),
            init_weights,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints
        )
    
    return result.x

In [ ]:
# Find optimal portfolios
max_sharpe_weights = optimize_portfolio(returns_df, objective='sharpe')
min_vol_weights = optimize_portfolio(returns_df, objective='min_vol')

# Display results
print("="*60)
print("MAXIMUM SHARPE RATIO PORTFOLIO")
print("="*60)
for name, weight in zip(asset_names, max_sharpe_weights):
    print(f"  {name:12s}: {weight:6.1%}")
print(f"\n  Expected Return: {portfolio_return(max_sharpe_weights, returns_df):.2%}")
print(f"  Volatility:      {portfolio_volatility(max_sharpe_weights, returns_df):.2%}")
print(f"  Sharpe Ratio:    {portfolio_sharpe(max_sharpe_weights, returns_df):.3f}")

print("\n" + "="*60)
print("MINIMUM VOLATILITY PORTFOLIO")
print("="*60)
for name, weight in zip(asset_names, min_vol_weights):
    print(f"  {name:12s}: {weight:6.1%}")
print(f"\n  Expected Return: {portfolio_return(min_vol_weights, returns_df):.2%}")
print(f"  Volatility:      {portfolio_volatility(min_vol_weights, returns_df):.2%}")
print(f"  Sharpe Ratio:    {portfolio_sharpe(min_vol_weights, returns_df):.3f}")

In [ ]:
# Compute efficient frontier
target_returns = np.linspace(0.06, 0.14, 50)
efficient_vols = []

for target in target_returns:
    try:
        weights = optimize_portfolio(returns_df, objective='target', target_return=target)
        efficient_vols.append(portfolio_volatility(weights, returns_df))
    except:
        efficient_vols.append(np.nan)

# Plot
plt.figure(figsize=(12, 8))

# Random portfolios
plt.scatter(results[:, 1], results[:, 0], c=results[:, 2], cmap='viridis', alpha=0.3, s=5)

# Efficient frontier
plt.plot(efficient_vols, target_returns, 'r-', linewidth=3, label='Efficient Frontier')

# Mark optimal portfolios
max_sharpe_ret = portfolio_return(max_sharpe_weights, returns_df)
max_sharpe_vol = portfolio_volatility(max_sharpe_weights, returns_df)
min_vol_ret = portfolio_return(min_vol_weights, returns_df)
min_vol_vol = portfolio_volatility(min_vol_weights, returns_df)

plt.scatter(max_sharpe_vol, max_sharpe_ret, marker='*', s=500, c='gold', 
            edgecolors='black', linewidths=2, label='Max Sharpe', zorder=5)
plt.scatter(min_vol_vol, min_vol_ret, marker='s', s=200, c='lime', 
            edgecolors='black', linewidths=2, label='Min Volatility', zorder=5)

# Capital Market Line (if risk-free rate = 2%)
rf = 0.02
cml_x = np.linspace(0, 0.3, 100)
cml_y = rf + (max_sharpe_ret - rf) / max_sharpe_vol * cml_x
plt.plot(cml_x, cml_y, 'b--', linewidth=2, label='Capital Market Line')

plt.title('Efficient Frontier with Optimal Portfolios')
plt.xlabel('Volatility (Annual)')
plt.ylabel('Expected Return (Annual)')
plt.xlim(0.05, 0.35)
plt.ylim(0, 0.18)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.colorbar(label='Sharpe Ratio')
plt.show()

In [ ]:
# Visualize allocation pie charts
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Equal weight
axes[0].pie(equal_weights, labels=asset_names, autopct='%1.0f%%', startangle=90)
axes[0].set_title(f'Equal Weight\nSharpe: {portfolio_sharpe(equal_weights, returns_df):.2f}')

# Max Sharpe
axes[1].pie(max_sharpe_weights, labels=asset_names, autopct='%1.0f%%', startangle=90)
axes[1].set_title(f'Max Sharpe\nSharpe: {portfolio_sharpe(max_sharpe_weights, returns_df):.2f}')

# Min Vol
axes[2].pie(min_vol_weights, labels=asset_names, autopct='%1.0f%%', startangle=90)
axes[2].set_title(f'Min Volatility\nSharpe: {portfolio_sharpe(min_vol_weights, returns_df):.2f}')

plt.tight_layout()
plt.show()

In [ ]:
%%ai $MODEL
Based on the portfolio optimization results:

1. Why might an investor choose the minimum volatility portfolio over max Sharpe?
2. What are the limitations of mean-variance optimization in practice?
3. How do transaction costs and rebalancing affect real-world implementation?

---

## Summary

This notebook demonstrated:
- Simulating correlated asset returns
- Calculating portfolio metrics (return, volatility, Sharpe ratio)
- Generating the efficient frontier
- Finding optimal portfolios (max Sharpe, min volatility)
- Visualizing portfolio allocations

**Next:** Explore options pricing in `02-options-pricing.ipynb`

---